# Self-RePrompt LoRA 推理交互测试

**用途**：加载任意一个训练好的 LoRA adapter，输入自定义问题，观察模型是否输出 `<SRP_START>…<SRP_END>` 格式。

**快速开始**：
1. 修改 Cell 1 中的 `ADAPTER_DIR`
2. 依次运行所有 Cell
3. 在最后一个 Cell 输入你的问题

In [1]:
# ── Cell 1：配置路径和设备 ────────────────────────────────────────
BASE_MODEL  = "../model/Qwen3-8B"                        # base 模型路径
ADAPTER_DIR = "../outputs/qwen3_sr_lora_hotpot_v2"       # LoRA adapter 路径
DEVICE      = "cuda:0"                                 # 使用的 GPU
MAX_NEW_TOKENS = 512                                   # 最大生成 token 数

In [2]:
# ── Cell 2：加载模型（只需运行一次）──────────────────────────────
import sys, os, textwrap
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# 确保项目根目录在 path 里
ROOT = os.path.abspath("..")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

SRP_START = "<SRP_START>"
SRP_END   = "<SRP_END>"

print("[1/3] 加载 tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
print(f"      词表大小: {len(tokenizer)}  "
      f"SRP_START id={tokenizer.convert_tokens_to_ids(SRP_START)}  "
      f"SRP_END id={tokenizer.convert_tokens_to_ids(SRP_END)}")

print("[2/3] 加载 base model ...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, dtype=torch.bfloat16,
    device_map=DEVICE, trust_remote_code=True,
)
base.resize_token_embeddings(len(tokenizer))  # 对齐 adapter 词表大小

print("[3/3] 挂载 LoRA adapter ...")
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()
print("✅ 模型加载完成，可以开始推理")

/data3/yyy/miniconda3/envs/srp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[1/3] 加载 tokenizer ...
      词表大小: 151671  SRP_START id=151669  SRP_END id=151670
[2/3] 加载 base model ...


Loading weights: 100%|██████████| 399/399 [00:04<00:00, 86.95it/s]


[3/3] 挂载 LoRA adapter ...


/data3/yyy/miniconda3/envs/srp/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


✅ 模型加载完成，可以开始推理


In [3]:
# ── Cell 3：推理函数定义 ──────────────────────────────────────────
def infer(user_input: str, max_new_tokens: int = MAX_NEW_TOKENS,
          do_sample: bool = False, temperature: float = 0.7,
          show_raw: bool = False) -> str:
    """
    对单条 user 输入进行推理，打印格式化结果。

    参数
    ----
    user_input     : 用户问题文本
    max_new_tokens : 最大生成 token 数
    do_sample      : 是否采样（True=随机，False=greedy）
    temperature    : 采样温度（do_sample=True 时生效）
    show_raw       : 是否额外打印原始 token 序列（调试用）
    """
    msgs   = [{"role": "user", "content": user_input}]
    prompt = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        pad_token_id=tokenizer.eos_token_id,
    )
    if do_sample:
        gen_kwargs["temperature"] = temperature

    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)

    new_ids  = out[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(new_ids, skip_special_tokens=False)

    # ── 解析 SRP 段 ──
    has_start = SRP_START in response
    has_end   = SRP_END   in response
    srp_ok    = has_start and has_end and response.index(SRP_START) < response.index(SRP_END)

    srp_content  = ""
    main_content = response
    if srp_ok:
        s = response.index(SRP_START) + len(SRP_START)
        e = response.index(SRP_END)
        srp_content  = response[s:e].strip()
        main_content = response[response.index(SRP_END) + len(SRP_END):].strip()
        # 去掉结尾的 <|im_end|>
        main_content = main_content.replace("<|im_end|>", "").strip()

    # ── 打印 ──
    sep = "─" * 68
    print(sep)
    print(f"📥 User: {user_input}")
    print(sep)
    if srp_ok:
        print(f"🧠 [SRP 策略段]")
        print(f"   {srp_content}")
        print()
        print(f"💬 [最终回答]")
        # 自动换行显示
        for line in main_content.split("\n"):
            print(textwrap.fill(line, width=80, subsequent_indent="   ") if line.strip() else "")
    else:
        print(f"⚠️  未检测到 SRP 格式 (SRP_START={has_start}, SRP_END={has_end})")
        print(response)
    print(sep)

    if show_raw:
        print("[raw response]")
        print(repr(response))

    return response

print("✅ infer() 函数已定义")

✅ infer() 函数已定义


In [6]:
# ── Cell 4：快速冒烟测试（预置例题）─────────────────────────────
# EXAMPLES = [
#     # HotpotQA 风格
#     "Were Scott Derrickson and Ed Wood from the same country?",
#     # GSM8K 风格
#     "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?",
#     # 开放域
#     "What is the difference between supervised and unsupervised learning?",
# ]
EXAMPLES =["What can you do for me?",
            "What nationality was James Henry Miller's wife?"
]
EXAMPLES =["Describe a time when you had to make a difficult decision."
]

EXAMPLES =["The sun is responsible for? A: puppies learning new tricks. B: children growing up and getting old. C: flowers wilting in a vase. D: plants sprouting, blooming and wilting."
]
for q in EXAMPLES:
    infer(q)
    print()

────────────────────────────────────────────────────────────────────
📥 User: The sun is responsible for? A: puppies learning new tricks. B: children growing up and getting old. C: flowers wilting in a vase. D: plants sprouting, blooming and wilting.
────────────────────────────────────────────────────────────────────
🧠 [SRP 策略段]
   Identify the direct relationship between the subject and the listed outcomes.

💬 [最终回答]
Let's examine each option for a direct relationship with the sun's role in
   natural processes.

**A: puppies learning new tricks**
This is a learned behavior in animals, primarily influenced by training, social
   interaction, and brain development, not directly caused by the sun.
**Not a direct relationship.**

**B: children growing up and getting old**
This is a biological aging process driven by genetics, nutrition, and time, not
   a direct effect of the sun.
**Not a direct relationship.**

**C: flowers wilting in a vase**
Wilting in a vase is due to lack of water a

In [ ]:
# ── Cell 5：自定义输入（修改这里的字符串后运行即可）────────────
MY_QUESTION = """
请在这里输入你的问题
""".strip()

infer(
    MY_QUESTION,
    max_new_tokens=512,
    do_sample=False,     # 改为 True 可以随机采样
    # temperature=0.7,   # do_sample=True 时启用
)

────────────────────────────────────────────────────────────────────
📥 User: 请在这里输入你的问题
────────────────────────────────────────────────────────────────────
🧠 [SRP 策略段]
   请描述您需要解决的问题或提出的问题内容。

💬 [最终回答]
当然，请告诉我您需要解决的问题或提出的具体疑问，我会尽力为您解答。
────────────────────────────────────────────────────────────────────


'<think>\n\n</think>\n\n<SRP_START>\n请描述您需要解决的问题或提出的问题内容。\n<SRP_END>\n当然，请告诉我您需要解决的问题或提出的具体疑问，我会尽力为您解答。<|im_end|>'

In [ ]:
# ── Cell 6：批量测试（从文件读取）────────────────────────────────
import json

DATA_FILE   = "../data/srp_prompt_with_answer/hotpot_train_qa_2000_with_srp_answer.jsonl"
N_SAMPLES   = 5    # 测试前 N 条
SHOW_GOLD   = True # 是否打印 gold answer

samples = []
with open(DATA_FILE) as f:
    for i, line in enumerate(f):
        if i >= N_SAMPLES: break
        samples.append(json.loads(line))

for s in samples:
    infer(s["user"])
    if SHOW_GOLD:
        print(f"  📌 Gold sr_prompt : {s.get('sr_prompt', 'N/A')}")
        print(f"  📌 Gold answer    : {s.get('srp_answer', s.get('answer', 'N/A'))[:200]}")
    print()

────────────────────────────────────────────────────────────────────
📥 User: Which magazine was started first Arthur's Magazine or First for Women?
────────────────────────────────────────────────────────────────────
🧠 [SRP 策略段]
   Compare the founding dates of the two publications to determine which one was established earlier.

💬 [最终回答]
Let's compare the founding dates:

1. **Arthur's Magazine**
   - First published in **1845** (originally as *The Youth's Companion* in 1828,
   but *Arthur's Magazine* was a later series starting in 1845).

2. **First for Women**
   - First published in **1978**.

**Conclusion:**
**Arthur's Magazine** was started first — more than 30 years before *First for
   Women*.
────────────────────────────────────────────────────────────────────
  📌 Gold sr_prompt : Determine the founding dates of each publication and compare them chronologically.
  📌 Gold answer    : Let’s determine the founding dates for each magazine and compare them.

**1. *Arthur’s Magazin

In [ ]:
# ── Cell 7：对比有无 LoRA 的输出（需提前加载好 model）──────────
# 用同一个 base model 在 LoRA disable 状态下生成，与激活 LoRA 对比

COMPARE_Q = "Were Scott Derrickson and Ed Wood from the same country?"

print("===== 激活 LoRA（新代码训练）=====")
model.enable_adapter_layers()
infer(COMPARE_Q)

print()
print("===== 禁用 LoRA（纯 base model）=====")
model.disable_adapter_layers()
infer(COMPARE_Q)

# 恢复 LoRA
model.enable_adapter_layers()

===== 激活 LoRA（新代码训练）=====
────────────────────────────────────────────────────────────────────
📥 User: Were Scott Derrickson and Ed Wood from the same country?
────────────────────────────────────────────────────────────────────
🧠 [SRP 策略段]
   Determine the nationality of each individual, then compare the two results.

💬 [最终回答]
Let’s break this down step by step.

**1. Scott Derrickson**
Scott Derrickson is an American film director, screenwriter, and producer. He
   was born in New York City, USA, and has worked primarily in the American film
   industry.
**Nationality:** American.

**2. Ed Wood**
Ed Wood was an American film director, producer, and screenwriter, best known
   for his low-budget films like *Plan 9 from Outer Space*. He was born in Fort
   Lauderdale, Florida, USA.
**Nationality:** American.

**3. Comparison**
Both individuals were born in the United States, worked in the American film
   industry, and are considered American by nationality.

**Answer:** Yes, Scott Der